Projet : MLOps — Prédiction du churn, détection de dérive et surveillance des performances 


**Environnement : VS Code · Python `.venv` · Git/GitHub · DVC**


# Partie I — Préparation des données, entraînement et évaluation du modèle V1

# Activer environnement virtuel
cd C:\Users\Hp\Projet_MLOps_Churn
.\.venv\Scripts\Activate.ps1

# Installer les bibliothèques Python
python -m pip install pandas numpy scikit-learn matplotlib seaborn joblib ipykernel evidently

pandas :	Importer, nettoyer et manipuler le dataset Churn.
numpy :	Effectuer des calculs numériques.
scikit-learn	: Diviser les données, entraîner et évaluer notre modèle de régression logistique.
matplotlib :	Créer des graphiques pour visualiser les données et les résultats.
seaborn :	Créer des graphiques statistiques, comme la matrice de confusion.
joblib	: Sauvegarder et charger notre modèle model_v1.pkl.
Evidently AI : servira à détecter les changements dans les données (Data Drift) et à surveiller les performances du modèle.

## Définir les chemins et préparer les dossiers

 Les chemins sont relatifs à la racine du projet : le notebook fonctionne dans VS Code, sans Colab. Exécuter les cellules dans l'ordre.

In [1]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, classification_report, ConfusionMatrixDisplay)

SEED = 42
# Recherche la racine du projet même si VS Code démarre dans notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'params.yaml').exists() and (ROOT.parent / 'params.yaml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'params.yaml').exists(), 'Ouvrir le dossier Projet_MLOps_Churn dans VS Code.'
DATA = ROOT / 'data'
MODELS = ROOT / 'models'
REFERENCES = ROOT / 'references'
REPORTS = ROOT / 'reports'
for dossier in (DATA, MODELS, REFERENCES, REPORTS):
    dossier.mkdir(parents=True, exist_ok=True)
print('Racine du projet :', ROOT)

Matplotlib is building the font cache; this may take a moment.


Racine du projet : C:\Users\Hp\Projet_MLOps_Churn


data/ : les données nettoyées et les quatre ensembles Train (60 %), Validation (10 %), Test (10 %) et Simulation (20 %).


models/ : le modèle de prédiction du churn entraîné et sauvegardé sous model_v1.pkl.


references/ : les données de référence T0, les distributions initiales des variables, les prédictions, les vraies étiquettes et les métriques de performance initiales, nécessaires à la future surveillance avec Evidently AI.


reports/ : les graphiques d'analyse des données, la matrice de confusion, les résultats de recherche d'hyperparamètres et les rapports d'évaluation du modèle.

## Étape 1 — Importer et explorer le dataset


Source :IBM Telco Customer Churn (lien Kaggle communiqué).
     Télécharger le CSV et le placer dans `data/WA_Fn-UseC_-Telco-Customer-Churn.csv`.

Pourquoi ? Vérifier les colonnes, types, dimensions, valeurs manquantes et proportions de churn avant tout apprentissage.

In [2]:
FICHIER = DATA / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
if not FICHIER.exists():
    raise FileNotFoundError(f'Dataset absent : {FICHIER}\nTélécharger le CSV depuis Kaggle puis le placer dans data/.')
df_brut = pd.read_csv(FICHIER)
print('Dimensions initiales :', df_brut.shape)
display(df_brut.head())
print('Types :')
display(df_brut.dtypes.to_frame('type'))
print('Valeurs manquantes explicites :')
display(df_brut.isna().sum().to_frame('manquants'))
print('Cible Churn :')
display(df_brut['Churn'].value_counts(dropna=False))

Dimensions initiales : (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Types :


,type
customerID,str
gender,str
SeniorCitizen,int64
Partner,str
Dependents,str
tenure,int64
PhoneService,str
MultipleLines,str
InternetService,str
OnlineSecurity,str


Valeurs manquantes explicites :


,manquants
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


Cible Churn :


Churn
No     5174
Yes    1869
Name: count, dtype: int64

### 3.1. Nettoyer les données
- Retirer `customerID` : identifiant, pas une caractéristique d'apprentissage.
- Convertir `TotalCharges` en numérique ; supprimer les lignes où la conversion est impossible.
- Transformer `Churn` : `No → 0`, `Yes → 1`.
- Conserver les noms de variables normalisés de l'ancien notebook pour les futurs composants.

In [ ]:
renommage = {
 'gender':'Gender', 'SeniorCitizen':'Senior_Citizen', 'tenure':'Tenure',
 'PhoneService':'Phone_Service', 'MultipleLines':'Multiple_Lines',
 'InternetService':'Internet_Service', 'OnlineSecurity':'Online_Security',
 'OnlineBackup':'Online_Backup', 'DeviceProtection':'Device_Protection',
 'TechSupport':'Tech_Support', 'StreamingTV':'Streaming_TV',
 'StreamingMovies':'Streaming_Movies', 'PaperlessBilling':'Paperless_Billing',
 'PaymentMethod':'Payment_Method', 'MonthlyCharges':'Monthly_Charges',
 'TotalCharges':'Total_Charges'
}
df = df_brut.rename(columns=renommage).copy()
df['Total_Charges'] = pd.to_numeric(df['Total_Charges'], errors='coerce')
print('Lignes avec Total_Charges manquant :', df['Total_Charges'].isna().sum())
df = df.dropna(subset=['Total_Charges']).drop(columns=['customerID']).copy()
df['Churn'] = df['Churn'].map({'No': 0, 'Yes': 1})
assert df['Churn'].notna().all(), 'Valeur inconnue dans Churn.'
df['Churn'] = df['Churn'].astype('int64')
assert not df.isna().any().any(), 'Des valeurs manquantes subsistent.'
assert df.shape[1] == 20, 'Vérifier les colonnes attendues (19 variables + Churn).'
print('Dimensions après nettoyage :', df.shape)
display(df.head())

### 3.2. Exploration descriptive
**Pourquoi ?** Comprendre la répartition du churn et sa relation descriptive avec le type de contrat. Ces graphiques ne démontrent pas une causalité.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x='Churn', ax=axes[0])
axes[0].set_title('Répartition de Churn (0 = Non, 1 = Oui)')
sns.countplot(data=df, x='Contract', hue='Churn', ax=axes[1])
axes[1].set_title('Churn selon le type de contrat')
axes[1].tick_params(axis='x', rotation=15)
plt.tight_layout()
fig.savefig(REPORTS / 'exploration_churn.png', dpi=160, bbox_inches='tight')
plt.show()

## Étape 4 — Répartir les données (60 % / 10 % / 10 % / 20 %)
**Pourquoi ?** Séparer apprentissage, validation, évaluation indépendante et simulation. La stratification maintient approximativement la proportion de churn dans chaque ensemble. La simulation est réservée aux étapes de monitoring : **elle ne sert pas à entraîner V1**.

In [ ]:
# 80 % historique, 20 % simulation.
historique, simulation_df = train_test_split(
    df, test_size=0.20, stratify=df['Churn'], random_state=SEED)
# 60 % Train (75 % de l'historique), 20 % restant.
train_df, reste = train_test_split(
    historique, test_size=0.25, stratify=historique['Churn'], random_state=SEED)
# 10 % Validation et 10 % Test.
validation_df, test_df = train_test_split(
    reste, test_size=0.50, stratify=reste['Churn'], random_state=SEED)
ensembles = {'train': train_df, 'validation': validation_df,
             'test': test_df, 'simulation': simulation_df}
resume = pd.DataFrame([{'ensemble': nom, 'lignes': len(part),
                        'pourcentage': round(100 * len(part) / len(df), 2),
                        'churn_%': round(100 * part['Churn'].mean(), 2)}
                       for nom, part in ensembles.items()])
display(resume)
assert sum(map(len, ensembles.values())) == len(df)
assert len(set().union(*(set(part.index) for part in ensembles.values()))) == len(df)
print('OK : ensembles stratifiés, sans chevauchement.')

### 4.1. Générer les fichiers de données
**Fichiers créés automatiquement :** `dataset_nettoye.csv`, `train.csv`, `validation.csv`, `test.csv`, `simulation_T0.csv`, `repartition.csv`.

**Versionnement :** ces CSV seront suivis avec **DVC**, et non ajoutés directement à GitHub. Les commandes DVC/Git sont indiquées à la fin du notebook.

In [ ]:
df.to_csv(DATA / 'dataset_nettoye.csv', index=False)
for nom, part in ensembles.items():
    sortie = 'simulation_T0.csv' if nom == 'simulation' else f'{nom}.csv'
    part.to_csv(DATA / sortie, index=False)
resume.to_csv(DATA / 'repartition.csv', index=False)
print('Fichiers générés dans', DATA)
for p in sorted(DATA.glob('*.csv')):
    print(' -', p.name)

## Étape 5 — Développer le modèle V1
**Pourquoi un pipeline ?** Le `StandardScaler`, le `OneHotEncoder` et la régression logistique sont regroupés. Le prétraitement est ajusté uniquement sur les données d'entraînement, y compris pendant la validation croisée, ce qui limite les fuites de données.

In [ ]:
num_features = ['Tenure', 'Monthly_Charges', 'Total_Charges']
cat_features = ['Gender', 'Senior_Citizen', 'Partner', 'Dependents',
 'Phone_Service', 'Multiple_Lines', 'Internet_Service', 'Online_Security',
 'Online_Backup', 'Device_Protection', 'Tech_Support', 'Streaming_TV',
 'Streaming_Movies', 'Contract', 'Paperless_Billing', 'Payment_Method']
features = num_features + cat_features
assert set(features) == set(df.columns) - {'Churn'}
X_train, y_train = train_df[features], train_df['Churn']
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
])
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(solver='liblinear', class_weight='balanced', random_state=SEED))
])
print('Variables explicatives :', len(features), '| Numériques :', len(num_features),
      '| Catégorielles :', len(cat_features))

### 5.1. Recherche des hyperparamètres avec GridSearchCV
**Pourquoi ?** Comparer les pénalités L1/L2, les valeurs de `C` et `max_iter`, en utilisant une validation croisée stratifiée sur **Train uniquement**. Critère de sélection : F1-score de la classe churn.

In [ ]:
param_grid = {
    'model__penalty': ['l1', 'l2'],
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__max_iter': [100, 500, 1000]
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
grid_search = GridSearchCV(pipeline, param_grid=param_grid, scoring='f1',
                           cv=cv, n_jobs=-1, refit=True)
grid_search.fit(X_train, y_train)
modele_V1 = grid_search.best_estimator_
print('Meilleurs paramètres :', grid_search.best_params_)
print('Meilleur F1 moyen en validation croisée :', round(grid_search.best_score_, 4))
resultats_cv = pd.DataFrame(grid_search.cv_results_).sort_values('rank_test_score')
resultats_cv.to_csv(REPORTS / 'gridsearch_resultats_V1.csv', index=False)
display(resultats_cv[['params', 'mean_test_score', 'std_test_score']].head(10))

## Étape 6 — Évaluer et enregistrer les performances de V1
**Pourquoi ?** L'accuracy seule peut masquer les erreurs sur la classe churn. Calculer accuracy, précision, rappel, F1-score, ROC-AUC et matrice de confusion. **Le test sert de référence d'évaluation et ne doit pas être utilisé pour ajuster les hyperparamètres.**

In [ ]:
def evaluer(modele, part):
    X, y = part[features], part['Churn']
    pred = modele.predict(X)
    proba = modele.predict_proba(X)[:, 1]
    return {
        'accuracy': float(accuracy_score(y, pred)),
        'precision': float(precision_score(y, pred, zero_division=0)),
        'recall': float(recall_score(y, pred, zero_division=0)),
        'f1': float(f1_score(y, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y, proba))
    }
metriques = {nom: evaluer(modele_V1, part)
             for nom, part in [('train', train_df), ('validation', validation_df), ('test', test_df)]}
tableau_metriques = pd.DataFrame(metriques).T
display(tableau_metriques.round(4))
print('Rapport sur Test :')
print(classification_report(test_df['Churn'], modele_V1.predict(test_df[features]),
                            target_names=['Pas de churn', 'Churn'], zero_division=0))
tableau_metriques.to_csv(REFERENCES / 'tableau_metriques_V1.csv', index_label='ensemble')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(test_df['Churn'],
    modele_V1.predict(test_df[features]), display_labels=['Non', 'Oui'], ax=ax)
ax.set_title('Matrice de confusion — V1 sur Test')
plt.tight_layout()
fig.savefig(REPORTS / 'matrice_confusion_V1.png', dpi=160, bbox_inches='tight')
plt.show()

## Étape 7 — Sauvegarder V1 et créer les références T0
**Fichiers générés automatiquement :**
- `models/model_v1.pkl` : pipeline complet (prétraitement + modèle).
- `references/reference_T0.csv` : distributions des **19 variables d'entraînement**, sans cible.
- `references/reference_performance_test.csv` : vraies étiquettes, prédictions et probabilités sur Test.
- `references/baseline_metrics.json` : métriques initiales sur Test.
- `references/performances_initiales.json` : métriques Train / Validation / Test.
- `references/schema_features.json` : ordre et types des variables, pour FastAPI et Evidently.
- `references/tableau_metriques_V1.csv` : tableau récapitulatif.
- `reports/` : graphiques et résultats de recherche d'hyperparamètres.

**Pourquoi ?** Le futur script `monitor.py` pourra recharger exactement ces éléments sans réentraîner V1.

In [ ]:
joblib.dump(modele_V1, MODELS / 'model_v1.pkl')
train_df[features].to_csv(REFERENCES / 'reference_T0.csv', index=False)
reference_performance = test_df.copy()
reference_performance['prediction'] = modele_V1.predict(test_df[features])
reference_performance['prediction_proba'] = modele_V1.predict_proba(test_df[features])[:, 1]
reference_performance.to_csv(REFERENCES / 'reference_performance_test.csv', index=False)
with (REFERENCES / 'baseline_metrics.json').open('w', encoding='utf-8') as f:
    json.dump(metriques['test'], f, indent=2, ensure_ascii=False)
with (REFERENCES / 'performances_initiales.json').open('w', encoding='utf-8') as f:
    json.dump(metriques, f, indent=2, ensure_ascii=False)
with (REFERENCES / 'schema_features.json').open('w', encoding='utf-8') as f:
    json.dump({'features': features, 'numeriques': num_features,
               'categorielles': cat_features, 'cible': 'Churn',
               'classe_positive': 1, 'seed': SEED}, f, indent=2, ensure_ascii=False)
print('Modèle et références T0 sauvegardés.')

### 7.1. Vérifier les fichiers et le modèle rechargé
**Pourquoi ?** Contrôler que les fichiers existent et que le pipeline enregistré reproduit les prédictions.

In [ ]:
modele_recharge = joblib.load(MODELS / 'model_v1.pkl')
assert np.array_equal(modele_V1.predict(test_df[features]),
                      modele_recharge.predict(test_df[features]))
obligatoires = [
    DATA / 'dataset_nettoye.csv', DATA / 'train.csv', DATA / 'validation.csv',
    DATA / 'test.csv', DATA / 'simulation_T0.csv', DATA / 'repartition.csv',
    MODELS / 'model_v1.pkl', REFERENCES / 'reference_T0.csv',
    REFERENCES / 'reference_performance_test.csv', REFERENCES / 'baseline_metrics.json',
    REFERENCES / 'performances_initiales.json', REFERENCES / 'schema_features.json',
    REFERENCES / 'tableau_metriques_V1.csv', REPORTS / 'exploration_churn.png',
    REPORTS / 'matrice_confusion_V1.png', REPORTS / 'gridsearch_resultats_V1.csv'
]
for chemin in obligatoires:
    assert chemin.exists(), f'Fichier manquant : {chemin}'
    print('OK :', chemin.relative_to(ROOT))
print('Vérification terminée : V1 est prêt pour la surveillance.')

## Versionnement — À exécuter dans le terminal VS Code après la dernière cellule
**1. DVC : suivre les datasets, le modèle et les références produites.** Le dossier `data/` contient aussi le CSV original. Les fichiers de `reports/` sont de petite taille et peuvent être suivis directement par Git ; les références T0 sont suivies par DVC.
```powershell
python -m dvc add data
python -m dvc add models
python -m dvc add references
python -m dvc push
```
`dvc push` envoie les fichiers suivis vers le stockage Google Drive configuré (une authentification peut être demandée). **Vérifier la réussite du push avant de partager les métadonnées.**

**2. Git : publier le notebook, les rapports et les métadonnées DVC.** Si vous travaillez sur `main` et que votre équipe l'autorise :
```powershell
git status
git add notebooks/Partie_1_Modele_V1_MLOps_Churn.ipynb reports/ data.dvc models.dvc references.dvc .gitignore
git commit -m "Preparation des donnees et modele V1 avec DVC"
git push origin main
```
Si vous utilisez une branche de fonctionnalité, remplacer `main` par son nom et créer une Pull Request. **Ne pas faire `git add` sur les CSV ou `.pkl` directement.**

**3. Ton amie :** `git pull origin main` puis `python -m dvc pull` pour retrouver les fichiers versionnés. L'environnement `.venv` reste local.

**Note :** ne pas relancer `dvc add data`/`models`/`references` avant que les fichiers existent. Si DVC signale des fichiers déjà suivis par Git, les retirer de l'index Git avec `git rm --cached` avant de les ajouter à DVC.

## Synthèse et suite du projet
Nous avons préparé le dataset, créé quatre ensembles stratifiés, entraîné et évalué V1, puis sauvegardé le modèle et les références T0. **Prochaines étapes :** `api/main.py` (FastAPI), simulations T1/T2/T3, `src/monitor.py` (Evidently AI), GitHub Actions et réentraînement V2. Les simulations ne sont pas encore générées : elles seront construites à partir de `data/simulation_T0.csv` dans l'étape 10, afin de ne pas utiliser les observations futures pendant l'entraînement V1.